# 01 · EDA of the master file (`pre_training_data.xlsx`)

The master file holds one row per recorded call, for both models (built by `salud-propensity master`):
- **synthetic** rows from the master world (Salud's time frame, calls to 2026-11-18, 51/51 calibration statistics),
  split 80/20 at random **by patient** (`split` = train / test);
- **real** rows: Salud's 84 calls (`split` = real_test), used only for testing.

| Model | Rows | Columns | Target |
|---|---|---|---|
| Pre-call | `label_usable` | 11 history fields (`D.MODEL_PRE`) | `paid_30d` |
| Post-call | `label_usable` & `post_call_row` | 11 + 21 call fields | `paid_after_call_30d` |

**Privacy:** the real rows are Salud's calls. Only aggregates of real data are shown. Outputs are stripped on commit.

Sections: 1 contents · 2 time · 3 patients · 4 targets · 5 split check · 6 pre-call fields · 7 duplicates ·
8 post-call fields · 9 data-quality gaps · 10 synthetic vs real (adversarial) · 11 decisions.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from salud_pipeline.phase2.propensity import dictionary as D, modeling as M
from salud_pipeline.phase2.propensity.master import MASTER_XLSX, SCORES, outcome_group

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)
# Two series throughout: synthetic (slot 1, blue) and real (slot 2, orange); palette validated (CVD dE 24.7).
SYN, REAL, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": GRID,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
                     "grid.color": GRID, "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 11, "axes.titlecolor": INK, "legend.frameon": False, "font.size": 9})

m = pd.read_excel(MASTER_XLSX)
m["as_of"] = pd.to_datetime(m["as_of"])
NUM = D.MODEL_PRE + D.MODEL_POST_EXTRA + ["paid_on_call", "paid_30d", "paid_after_call_30d", "placement_balance",
      "days_since_last_discharge", "ever_paid", "amount_paid_90d", "amount_paid_365d", "n_payments_ever",
      "amount_paid_ever", "amount_insurance_365d", "amount_adjustments_365d"]
for c in NUM:
    m[c] = pd.to_numeric(m[c].astype(float) if m[c].dtype == bool else m[c], errors="coerce")
syn, real = m[m.data_type == "synthetic"], m[m.data_type == "real"]
su, ru = syn[syn.label_usable], real[real.label_usable]            # rows with a trustworthy 30-day label
sp, rp = su[su.post_call_row], ru[ru.post_call_row]                 # post-call rows
print(f"{len(m):,} rows x {m.shape[1]} columns")

## 1 · What is in the file

In [ ]:
tab = m.groupby(["data_type", "split"]).agg(
    calls=("call_id", "size"), patients=("guarantor_id", "nunique"), usable=("label_usable", "sum"),
    post_call_rows=("post_call_row", lambda s: int((s & m.loc[s.index, "label_usable"]).sum())))
tab

**Result:** synthetic train 6,971 calls (1,550 patients; 6,647 with a usable label, 5,368 post-call rows); synthetic test
1,778 calls (387 patients; 1,702 usable, 1,365 post-call); real 84 calls (77 patients; 58 usable, 47 post-call).

In [ ]:
groups = pd.Series({"row selection / bookkeeping": ["data_type", "split", "label_usable", "post_call_row", "sentiment_source", "qa_source"],
                    "pre-call model fields": D.MODEL_PRE, "post-call extra fields": D.MODEL_POST_EXTRA,
                    "targets": ["paid_30d", "paid_after_call_30d"]})
used = set(sum(groups.tolist(), []))
groups["other (ids, dates, other labels, dropped candidates)"] = [c for c in m.columns if c not in used]
groups.map(len).rename("columns").to_frame()

**Result:** 74 columns: 6 for selecting rows, 11 pre-call fields, 21 post-call extras, 2 targets, and 34 others
(ids, dates, other labels, candidates not in the current models).

## 2 · Time coverage

Synthetic calls run from May 2023 to 17 Nov 2026 at a steady volume. Real calls fall on four days (Jul 1, Oct 7–9 2026);
all 58 usable real rows are from Jul 1. Calls after Oct 19 have no complete 30-day window (`label_usable` = False).

In [ ]:
by_m = syn.groupby(syn.as_of.dt.to_period("M")).agg(calls=("call_id", "size"), usable=("label_usable", "mean"))
fig, ax = plt.subplots(figsize=(9, 2.8))
x = by_m.index.to_timestamp()
ax.plot(x, by_m["calls"], color=SYN, lw=2, label="synthetic calls per month")
for d, n in real.groupby(real.as_of.dt.normalize()).size().items():
    ax.axvline(d, color=REAL, lw=1, ls=":")
ax.text(pd.Timestamp("2026-07-01"), by_m["calls"].max() * 0.95, "  real call days", color=MUTED, fontsize=8)
ax.axvspan(pd.Timestamp("2026-10-19"), pd.Timestamp("2026-11-18"), color=GRID, alpha=0.8)
ax.text(pd.Timestamp("2026-10-20"), 10, "no full\n30 days", color=MUTED, fontsize=8)
ax.set_ylabel("calls"); ax.set_title("Synthetic calls per month (real call days dotted)"); ax.set_ylim(0)
plt.show()
print("synthetic calls by year:", syn.as_of.dt.year.value_counts().sort_index().to_dict())
print("unusable synthetic rows (window not complete):", int((~syn.window_complete).sum()))

## 3 · Patients: several calls each

Every synthetic patient has 1–15 calls (median 4), and repeat patients often have **mixed outcomes**. So rows of one
patient are not independent: any split or cross-validation must keep a patient's rows together (the `split` column
does; K-fold inside `train` must be grouped by `guarantor_id`).

In [ ]:
per = syn.groupby("guarantor_id").size()
g = su.groupby("guarantor_id")["paid_30d"].agg(["mean", "size"]); g = g[g["size"] > 1]
print("calls per synthetic patient:", per.describe()[["mean", "50%", "max"]].round(1).to_dict())
print(f"repeat patients with mixed outcomes (paid after some calls, not others): {((g['mean'] > 0) & (g['mean'] < 1)).mean():.0%}")
print("calls per real patient:", real.groupby("guarantor_id").size().describe()[["mean", "max"]].round(2).to_dict())

## 4 · Targets

In [ ]:
def rate_ci(y):
    y = y.dropna().astype(float); p = y.mean(); n = len(y); se = np.sqrt(p * (1 - p) / n)
    return pd.Series({"n": n, "rate": p, "95% low": max(p - 1.96 * se, 0), "95% high": min(p + 1.96 * se, 1)})
rows = {}
for name, d, t in [("pre-call: synthetic train", su[su.split == "train"], "paid_30d"),
                   ("pre-call: synthetic test", su[su.split == "test"], "paid_30d"),
                   ("pre-call: real", ru, "paid_30d"),
                   ("post-call: synthetic train", sp[sp.split == "train"], "paid_after_call_30d"),
                   ("post-call: synthetic test", sp[sp.split == "test"], "paid_after_call_30d"),
                   ("post-call: real", rp, "paid_after_call_30d")]:
    rows[name] = rate_ci(d[t])
targets = pd.DataFrame(rows).T; targets["n"] = targets["n"].astype(int); targets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
yr = su.groupby(su.as_of.dt.year)
axes[0].plot(yr["paid_30d"].mean().index, yr["paid_30d"].mean(), color=SYN, lw=2, marker="o", ms=5, label="pre-call (paid_30d)")
yp = sp.groupby(sp.as_of.dt.year)["paid_after_call_30d"].mean()
axes[0].plot(yp.index, yp, color=SYN, lw=2, ls="--", marker="o", ms=5, label="post-call (paid after call)")
axes[0].set_ylim(0, 1); axes[0].set_xticks(yp.index); axes[0].set_title("Synthetic pay rate by year"); axes[0].legend(fontsize=8)
bins = np.arange(0, 32, 2)
for d, col, lab in [(su, SYN, "synthetic"), (ru, REAL, "real")]:
    x = d.loc[d.paid_30d == 1, "days_to_first_payment"]
    axes[1].hist(x, bins=bins, density=True, histtype="step", lw=2, color=col, label=f"{lab} (n={len(x)})")
axes[1].set_title("Days from the call to the first payment (payers)"); axes[1].set_xlabel("days"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
for name, d in [("synthetic", su), ("real", ru)]:
    x = d.loc[d.paid_on_call == 1, "days_to_first_payment"]
    print(f"{name}: paid on the call {d.paid_on_call.mean():.0%} of rows; their payment is posted on day 0: {(x == 0).mean():.0%}, day 1: {(x == 1).mean():.0%}")

**Reading:** the synthetic pay rates (pre 0.56, post 0.45) sit a little above real (0.50, 0.36) but inside the real
intervals; 2023 is lower because patients have little history at the start of the world. Payment on the call happens
equally often (19%), and on-call payments post the same day in both.
Fixed: on-call payments now post the same day (`same_day_posting`).

## 5 · Split check: train and test look alike

In [ ]:
tr, te = su[su.split == "train"], su[su.split == "test"]
print("patients on both sides:", len(set(tr.guarantor_id) & set(te.guarantor_id)))
pd.DataFrame({"train": tr[D.MODEL_PRE + ["paid_30d"]].median(), "test": te[D.MODEL_PRE + ["paid_30d"]].median(),
              "train mean": tr[D.MODEL_PRE + ["paid_30d"]].mean(), "test mean": te[D.MODEL_PRE + ["paid_30d"]].mean()}).round(3)

**Result:** no patient is on both sides. Train and test medians match (e.g. 2 accounts, ~$440 balance, 85 vs 91
days since first placement) and the pay rate is 0.567 vs 0.570.

## 6 · Pre-call fields

Synthetic and real now share the same population (call days). Payment history is the strongest single field in both,
at a similar strength (about 0.68 synthetic, 0.72 real), unlike the earlier world where it was much stronger in synthetic.

In [ ]:
def summ(d, c):
    x = d[c]
    return [x.median(), (x == 0).mean(), x.isna().mean()]
t = pd.DataFrame({c: summ(su, c) + summ(ru, c) + [su[c].skew()] for c in D.MODEL_PRE},
                 index=pd.MultiIndex.from_product([["synthetic", "real"], ["median", "zero share", "missing"]]).append(
                     pd.MultiIndex.from_tuples([("synthetic", "skew")]))).T
t.round(2)

**Result:** synthetic and real medians are close (2 accounts each; balance $446 vs $566; days since first placement 86 vs
67; days since last payment 20 vs 19). Zero shares are similar (payments in 90 days: 67% vs 71% zero). Skew up to 5.6
in synthetic, so the models log-transform money and counts.

In [ ]:
show = ["balance_at_t", "days_since_first_placement", "days_since_last_placement", "n_accounts",
        "n_payments_365d", "days_since_last_payment"]
fig, axes = plt.subplots(2, 3, figsize=(10, 5))
for ax, c in zip(axes.ravel(), show):
    lo = 0 if c.startswith("n_") else 1
    a, b = su[c].dropna(), ru[c].dropna()
    edges = np.unique(np.quantile(np.log1p(pd.concat([a, b])), np.linspace(0, 1, 21)))
    ax.hist(np.log1p(a), bins=edges, density=True, histtype="step", lw=2, color=SYN, label="synthetic")
    ax.hist(np.log1p(b), bins=edges, density=True, histtype="step", lw=2, color=REAL, label="real (58)")
    ax.set_title(c, fontsize=9); ax.set_xlabel("log(1 + value)"); ax.set_yticks([])
axes[0, 0].legend(fontsize=8)
plt.suptitle("Pre-call fields: synthetic vs real (log scale; days_since_last_payment only for patients who paid)", fontsize=10, color=INK)
plt.tight_layout(); plt.show()

In [ ]:
def auc(d, c, t):
    x, y = d[c], d[t]
    x = x.fillna(x.max() + 1 if c == "days_since_last_payment" else 0)   # never paid = longest ago
    return roc_auc_score(y.astype(int), x) if x.nunique() > 1 and y.nunique() > 1 else np.nan
pre_auc = pd.DataFrame({"synthetic": [auc(su, c, "paid_30d") for c in D.MODEL_PRE],
                        "real (58)": [auc(ru, c, "paid_30d") for c in D.MODEL_PRE]}, index=D.MODEL_PRE)
fig, ax = plt.subplots(figsize=(7, 3.6))
y = np.arange(len(pre_auc))
ax.hlines(y, pre_auc.min(axis=1), pre_auc.max(axis=1), color=GRID, lw=2)
ax.plot(pre_auc["synthetic"], y, "o", ms=8, color=SYN, mec="#fcfcfb", mew=2, label="synthetic")
ax.plot(pre_auc["real (58)"], y, "o", ms=8, color=REAL, mec="#fcfcfb", mew=2, label="real (58)")
ax.axvline(0.5, color=MUTED, lw=1); ax.set_yticks(y, pre_auc.index); ax.invert_yaxis()
ax.set_xlabel("AUC of the field alone for paid_30d (0.5 = no signal; < 0.5 = higher value pays less)")
ax.legend(fontsize=8, loc="lower right"); ax.set_title("Each pre-call field on its own")
plt.tight_layout(); plt.show()
pre_auc.round(3)

**Reading:** directions agree for every field that carries signal (payment counts up, days since last payment down,
debt age up). Account counts point slightly opposite ways, but both are near 0.5. The real AUCs have a ±0.15
interval (58 rows), so only payment history is clearly a signal there.

## 7 · Duplicate fields (watch item 20)

The 11 fields come from 23 candidates after dropping near-duplicates (`modeling.DUPLICATES`, Spearman ρ > 0.9), a list
fixed on the original world. Re-checked on the master file:

In [ ]:
pairs = [(k, v) for k, vs in M.DUPLICATES.items() for v in vs if k in su and v in su] + [("n_payments_90d", "n_payments_365d")]
dup = pd.DataFrame([(a, b, su[[a, b]].corr("spearman").iloc[0, 1], ru[[a, b]].corr("spearman").iloc[0, 1]) for a, b in pairs],
                   columns=["kept", "dropped / compared", "rho synthetic", "rho real"]).round(2)
dup["rule (> 0.9) holds?"] = dup["rho synthetic"] > 0.9
dup

In [ ]:
cand = [c for c in M.FS_ALL if c in su and su[c].nunique() > 1]
corr = su[cand].corr("spearman")
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr.abs(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(cand)), cand, rotation=90, fontsize=7); ax.set_yticks(range(len(cand)), cand, fontsize=7)
ax.grid(False); plt.colorbar(im, ax=ax, shrink=0.7, label="|Spearman rho| (synthetic)")
ax.set_title("Correlation between the 23 pre-call candidates"); plt.tight_layout(); plt.show()

**Reading:** in the master world the amount fields are duplicates again (ρ 0.96–0.99), so dropping them is justified here.
`days_since_last_discharge` is **no longer** a duplicate (ρ 0.71), so it could be tried as a field. `n_payments_90d`
and `n_payments_365d` are 0.92 apart, over the rule's cut-off; both are kept because they measure different recency. `reviewed_balance` vs `action_shared_balance` is 0.40 in synthetic (0.95 real): the generator sets them independently; neither is a model field.

## 8 · Post-call fields

Call outcome fields are produced by the call simulation; sentiment and QA are **sampled from real calls with the same
outcome** (no transcripts), so they can only repeat what the outcome already says.

In [ ]:
calls = D.MODEL_POST_EXTRA
share = pd.DataFrame({"synthetic share": sp[calls].mean(), "real share (47)": rp[calls].mean(),
                      "synthetic AUC": [auc(sp, c, "paid_after_call_30d") for c in calls],
                      "real AUC (47)": [auc(rp, c, "paid_after_call_30d") for c in calls]})
share.round(3)

**Result:** call-field shares and directions are broadly similar; the largest gaps are inbound (78% vs 51% of post-call
rows) and transferred (38% vs 21%), both inside what the 84-call calibration allows. Promise to pay (synthetic AUC 0.58,
real 0.58) and payment plan (0.58 / 0.56) carry signal in both; sentiment and QA are near 0.5 in synthetic.

In [ ]:
grp = lambda d: d.assign(outcome=outcome_group(d)).groupby("outcome")["paid_after_call_30d"].agg(["mean", "size"])
o = grp(sp).join(grp(rp), lsuffix=" synthetic", rsuffix=" real").fillna(0)
o = o.sort_values("mean synthetic")
fig, ax = plt.subplots(figsize=(7, 3))
y = np.arange(len(o))
ax.barh(y + 0.2, o["mean synthetic"], height=0.38, color=SYN, label="synthetic")
ax.barh(y - 0.2, o["mean real"], height=0.38, color=REAL, label="real (47)")
for i, (n_s, n_r) in enumerate(zip(o["size synthetic"], o["size real"])):
    ax.text(1.01, i, f"n={int(n_s)} / {int(n_r)}", va="center", fontsize=7, color=MUTED, transform=ax.get_yaxis_transform())
ax.set_yticks(y, o.index); ax.set_xlim(0, 1); ax.set_xlabel("paid in the 30 days after the call")
ax.legend(fontsize=8, loc="lower right"); ax.set_title("Pay rate after the call, by how the call ended")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
for ax, c in zip(axes, SCORES):
    a, b = sp[c].dropna(), rp[c].dropna()
    edges = np.linspace(min(a.min(), b.min()), max(a.max(), b.max()), 16)
    ax.hist(a, bins=edges, density=True, histtype="step", lw=2, color=SYN, label="synthetic")
    ax.hist(b, bins=edges, density=True, histtype="step", lw=2, color=REAL, label="real (47)")
    ax.set_title(c.replace("_sentiment", "").replace("_", " "), fontsize=9); ax.set_yticks([])
axes[0].legend(fontsize=7); plt.suptitle("Sentiment and QA: synthetic (sampled) vs real", fontsize=10, color=INK)
plt.tight_layout(); plt.show()
print("QA values: real", sorted(rp.qa_score.round(3).unique()), "| synthetic distinct:", sp.qa_score.nunique())

**Reading:** promise to pay and payment plans raise the chance of paying later in both; sentiment and QA carry
almost no signal in synthetic (AUC about 0.48–0.53, by construction) and a weak one in real (0.52–0.59, ±0.17 on 47 rows).
QA takes only the real score values (items passed / items applicable).

## 9 · Call fields: synthetic vs real, after the fixes

The first EDA pass found gaps here; the generator now fills these fields the way Salud does (master config:
`salud_collection_attempt`, `asked_fa_rate`, `insurance_review_p`, `same_day_posting`), and four statistics were added to
`salud-synth-calibrate` (now 55/55). The insurance flag no longer counts "Explained Coinsurance/Deductible/Copay".

In [ ]:
raw = lambda d: pd.Series({
    "collection attempted = yes, agent calls": d.loc[d.spoke_with_agent == 1, "collection_attempted"].mean(),
    "collection attempted = yes, phone-menu calls": d.loc[d.spoke_with_agent == 0, "collection_attempted"].mean(),
    "asked about financial assistance": d["asked_financial_assistance"].mean(),
    "action: insurance": d["action_insurance"].mean(),
    "talk seconds, agent calls (median)": d.loc[d.spoke_with_agent == 1, "talk_seconds"].median(),
    "talk seconds, phone-menu calls (median)": d.loc[d.spoke_with_agent == 0, "talk_seconds"].median()})
pd.DataFrame({"synthetic (all calls)": raw(syn), "real (84 calls)": raw(real)}).round(3)

**Result:** after the fixes, the call fields follow Salud's practice: collection attempt "yes" on 4.0% of synthetic
vs 4.5% of real agent calls; asked about financial assistance 17.4% vs 14.3%; insurance action 1.4% vs 1.2%; median talk
time 224 vs 229 s (agent calls) and 30 vs 26 s (phone menu).

| Field | Salud's practice | Synthetic now |
|---|---|---|
| `collection_attempted` | blank on ~96% of agent calls (blank reads as "no"), "Yes" on phone-menu calls | same |
| `asked_financial_assistance` | ~18% of agent calls | `asked_fa_rate` 0.17 |
| `action_insurance` | rare (insurance work only) | `insurance_review_p` 0.1 |
| money taken on a call | posts the same day | `same_day_posting` |

Because Salud leaves it blank, `collection_attempted` only repeats "phone-menu call" (`spoke_with_agent` = 0) in both worlds.
Payment-plan actions in real data are mostly "Update Payment Plan" (hospital-side plans, watch item 24); synthetic sets plans up.

## 10 · How different are synthetic and real rows? (adversarial validation)

A classifier tries to tell real rows from synthetic ones; AUC 0.5 = indistinguishable. Permutation shows what gives
them away. (The earlier world scored 0.945 on the pre-call fields, mostly from contact-history fields.)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
def adversarial(F, seed=0):
    d = pd.concat([su.sample(2000, random_state=seed), ru]); y = (d.data_type == "real").astype(int); X = d[F].fillna(-1)
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    clf = HistGradientBoostingClassifier(max_depth=3, random_state=seed)
    base = roc_auc_score(y, cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:, 1])
    drop = {}
    for c in F:
        Xp = X.copy(); Xp[c] = np.random.default_rng(seed).permutation(Xp[c].values)
        drop[c] = base - roc_auc_score(y, cross_val_predict(clf, Xp, y, cv=cv, method="predict_proba")[:, 1])
    return base, pd.Series(drop).sort_values(ascending=False)
for name, F in [("pre-call (11)", D.MODEL_PRE), ("post-call (32)", D.MODEL_PRE + D.MODEL_POST_EXTRA),
                ("post-call without collection_attempted (31)", D.MODEL_PRE + [c for c in D.MODEL_POST_EXTRA if c != "collection_attempted"])]:
    a, top = adversarial(F)
    print(f"{name}: adversarial AUC {a:.3f}; gives it away: " + ", ".join(f"{k} {v:+.3f}" for k, v in top.head(4).items()))

**Result:** adversarial AUC 0.845 on the pre-call fields (debt-age fields give it away: all real rows are from one
day), 0.877 on the post-call fields (down from 0.962 before the call fields were made to match Salud's practice).

**Reading:** the pre-call fields are still distinguishable, mostly by placement timing (all real rows come from one
day, 2026-07-01, so their debt ages cluster). With 58 real rows this is a direction, not a measurement.

## 11 · Decisions for modelling on the master file

| Topic | Decision |
|---|---|
| Rows | `label_usable` only; post-call adds `post_call_row` |
| Train / test | synthetic `split`; K-fold inside train **grouped by patient**; real rows test only |
| Pre-call fields | the 11 (`D.MODEL_PRE`); amounts stay dropped (duplicates here, ρ ≥ 0.96); `days_since_last_discharge` may be tried |
| Post-call fields | the 21 extras; `collection_attempted` now only repeats `spoke_with_agent` (Salud leaves it blank), so it can be dropped without loss |
| Transforms | log1p for counts, balances and days (skew up to ~7); "never paid" fill + flag for `days_since_last_payment` (60% / 71% missing); trees need none |
| Class balance | pay rate 0.45–0.56: no resampling; stratify folds |
| Sentiment / QA | can't show value beyond the outcome on synthetic data (sampled by outcome); judge them on real calls only |
| Data checks | `salud-propensity master` refuses to write the file unless 24 integrity checks pass; an independent recomputation from the raw tables matched every field on all real and 600 synthetic rows |
| Known limits | Salud-side history only (hospital plan payments missing, watch item 24); balance ignores later charges (item 25) |